In [ ]:
import numpy as np
import pandas as pd
import glob
import os
import io
import re

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Dictionary to store datasets
dataframes = {}

# Read each uploaded file
for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    dataframes[filename] = df
    print(f"\n--- {filename} ---")
    print(df.head())  # Print first 5 rows of each file

In [ ]:
df = pd.read_csv('marketshare-20-80.csv')
print(df.info())
print(df.head())
df

In [ ]:
# Store cleaned DataFrames
cleaned_data = []
all_data = {}

In [ ]:
def parse_variable_components(df):
    # Split variable and extract bracket content
    df['Variable_Type'] = df['Variable'].str.extract(r'^([^\[]+)', expand=False)
    df['Bracket_Content'] = df['Variable'].str.extract(r'\[(.*?)\]', expand=False)
    df['Component_List'] = df['Bracket_Content'].str.split(',')

    # Initialize new columns
    df['Component_Count'] = df['Component_List'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    df['Year'] = pd.NA
    df['Stage'] = pd.NA
    df['Tech_In'] = pd.NA
    df['Tech_Out'] = pd.NA
    df['Region'] = pd.NA
    df['Region_From'] = pd.NA
    df['Region_To'] = pd.NA

    # Efficient row-wise parsing
    def parse_components(row):
        comps = row['Component_List']
        if not isinstance(comps, list):
            return row

        if len(comps) == 1:
            row['Region'] = comps[0].strip()
        elif len(comps) == 4:
            row['Year'] = pd.to_numeric(comps[0].strip(), errors='coerce')
            row['Stage'] = comps[1].strip()
            row['Tech_Out'] = comps[2].strip()
            row['Region'] = comps[3].strip()
            row['Tech_In'] = 'Earth' if 'mine' in row['Variable_Type'] else comps[2].strip()
        elif len(comps) == 5:
            row['Year'] = pd.to_numeric(comps[0].strip(), errors='coerce')
            row['Stage'] = comps[1].strip()
            if row['Variable_Type'].startswith(('import', 'export')):
                row['Tech_Out'] = comps[2].strip()
                row['Region_From'] = comps[3].strip()
                row['Region_To'] = comps[4].strip()
            else:
                row['Tech_In'] = comps[2].strip()
                row['Tech_Out'] = comps[3].strip()
                row['Region'] = comps[4].strip()

        return row

    df = df.apply(parse_components, axis=1)
    return df.drop(['Bracket_Content', 'Component_List'], axis=1)

In [ ]:
def process_file(filename, content):
    df = pd.read_csv(io.BytesIO(content))

    # Standardize column names
    df = df.rename(columns={'Variable Name': 'Variable', 'Solution Value': 'Value'})

    # Drop invalid or blank rows
    df = df.dropna(subset=['Variable'])
    df = df[df['Variable'].str.strip() != '']

    df['Source_File'] = filename

    # Parse components
    df = parse_variable_components(df)

    cleaned_data.append(df)
    all_data[filename] = df
    print(f"✅ Cleaned: {filename} → {df.shape[0]} rows")
    return df

In [ ]:
# Process all uploaded files
for filename, content in uploaded.items():
    process_file(filename, content)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

# Dictionary to store datasets
dataframes = {}

# Read each uploaded file
for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    dataframes[filename] = df
    print(f"\n--- {filename} ---")
    print(df.head())

# Store cleaned DataFrames
cleaned_data = []
all_data = {}

In [ ]:
def parse_variable_components(df):
    # Split variable and extract bracket content
    df['Variable_Type'] = df['Variable'].str.extract(r'^([^\[]+)', expand=False)
    df['Bracket_Content'] = df['Variable'].str.extract(r'\[(.*?)\]', expand=False)
    df['Component_List'] = df['Bracket_Content'].str.split(',')

    # Initialize new columns
    df['Component_Count'] = df['Component_List'].apply(lambda x: len(x) if isinstance(x, list) else 0)
    df['Year'] = pd.NA
    df['Stage'] = pd.NA
    df['Tech_In'] = pd.NA
    df['Tech_Out'] = pd.NA
    df['Region'] = pd.NA
    df['Region_From'] = pd.NA
    df['Region_To'] = pd.NA

    # Efficient row-wise parsing
    def parse_components(row):
        comps = row['Component_List']
        if not isinstance(comps, list):
            return row

        if len(comps) == 1:
            row['Region'] = comps[0].strip()
        elif len(comps) == 4:
            row['Year'] = pd.to_numeric(comps[0].strip(), errors='coerce')
            row['Stage'] = comps[1].strip()
            row['Tech_Out'] = comps[2].strip()
            row['Region'] = comps[3].strip()
            row['Tech_In'] = 'Earth' if 'mine' in row['Variable_Type'] else comps[2].strip()
        elif len(comps) == 5:
            row['Year'] = pd.to_numeric(comps[0].strip(), errors='coerce')
            row['Stage'] = comps[1].strip()
            if row['Variable_Type'].startswith(('import', 'export')):
                row['Tech_Out'] = comps[2].strip()
                row['Region_From'] = comps[3].strip()
                row['Region_To'] = comps[4].strip()
            else:
                row['Tech_In'] = comps[2].strip()
                row['Tech_Out'] = comps[3].strip()
                row['Region'] = comps[4].strip()

        return row

    df = df.apply(parse_components, axis=1)
    return df.drop(['Bracket_Content', 'Component_List'], axis=1)

def process_file(filename, content):
    df = pd.read_csv(io.BytesIO(content))
    # Standardize column names
    df = df.rename(columns={'Variable Name': 'Variable', 'Solution Value': 'Value'})
    # Drop invalid or blank rows
    df = df.dropna(subset=['Variable'])
    df = df[df['Variable'].str.strip() != '']
    df['Source_File'] = filename
    # Parse components
    df = parse_variable_components(df)
    cleaned_data.append(df)
    all_data[filename] = df
    print(f"✅ Cleaned: {filename} → {df.shape[0]} rows")
    return df


## Data Visualization

In [ ]:
# Process all uploaded files
for filename, content in uploaded.items():
    process_file(filename, content)

# INDIVIDUAL ANALYSIS FOR EACH DATASET
analysis_results = {}

for filename, df in all_data.items():
    print(f"\n{'='*60}")
    print(f"📊 ANALYZING: {filename}")
    print(f"{'='*60}")

    # 1. TOTAL PRODUCTION BY STAGE AND REGION THROUGHOUT YEARS
    prod_data = df[df['Variable_Type'].str.startswith('prod_') & df['Year'].notna()]

    if not prod_data.empty:
        # Production by stage, region, and year
        production_summary = prod_data.groupby(['Stage', 'Region', 'Year'])['Value'].sum().reset_index()
        production_pivot = production_summary.pivot_table(
            index=['Stage', 'Region'],
            columns='Year',
            values='Value',
            fill_value=0
        )

        print("\n📈 PRODUCTION BY STAGE AND REGION:")
        print(production_pivot)

        # Total production by stage (all years combined)
        stage_totals = prod_data.groupby('Stage')['Value'].sum().sort_values(ascending=False)
        print(f"\n📊 TOTAL PRODUCTION BY STAGE:")
        for stage, total in stage_totals.items():
            print(f"  {stage}: {total:,.2f}")
             # 2. ACTIVE FACILITIES BY STAGE AND REGION THROUGHOUT YEARS
    active_data = df[df['Variable_Type'].str.startswith('active_') & df['Year'].notna()]

    if not active_data.empty:
        # Active facilities by stage, region, and year
        active_summary = active_data.groupby(['Stage', 'Region', 'Year'])['Value'].sum().reset_index()
        active_pivot = active_summary.pivot_table(
            index=['Stage', 'Region'],
            columns='Year',
            values='Value',
            fill_value=0
        )

        print(f"\n🏭 ACTIVE FACILITIES BY STAGE AND REGION:")
        print(active_pivot)

        # Total active facilities by stage (all years combined)
        active_stage_totals = active_data.groupby('Stage')['Value'].sum().sort_values(ascending=False)
        print(f"\n🔢 TOTAL ACTIVE FACILITIES BY STAGE:")
        for stage, total in active_stage_totals.items():
            print(f"  {stage}: {total:,.0f}")
  # 3. PRODUCTION BY STAGE FOR EACH REGION
    if not prod_data.empty:
        production_by_stage_region = prod_data.groupby(['Region', 'Stage'])['Value'].sum().reset_index()
        production_stage_pivot = production_by_stage_region.pivot(
            index='Region',
            columns='Stage',
            values='Value'
        ).fillna(0)

        print(f"\n🌍 PRODUCTION BY STAGE FOR EACH REGION:")
        print(production_stage_pivot)

    # 4. ACTIVE FACILITIES BY STAGE FOR EACH REGION
    if not active_data.empty:
        active_by_stage_region = active_data.groupby(['Region', 'Stage'])['Value'].sum().reset_index()
        active_stage_pivot = active_by_stage_region.pivot(
            index='Region',
            columns='Stage',
            values='Value'
        ).fillna(0)

        print(f"\n🏢 ACTIVE FACILITIES BY STAGE FOR EACH REGION:")
        print(active_stage_pivot)
        # Store results for comparison
        analysis_results[filename] = {
        'production_by_stage': stage_totals if not prod_data.empty else pd.Series(),
        'active_by_stage': active_stage_totals if not active_data.empty else pd.Series(),
        'production_stage_pivot': production_stage_pivot if not prod_data.empty else pd.DataFrame(),
        'active_stage_pivot': active_stage_pivot if not active_data.empty else pd.DataFrame(),
        'production_summary': production_summary if not prod_data.empty else pd.DataFrame(),
        'active_summary': active_summary if not active_data.empty else pd.DataFrame()
    }

In [ ]:
# Save to Excel with multiple sheets
clean_filename = filename.replace('.csv', '').replace('-', '_').replace(' ', '_')
excel_file = f'{clean_filename}_analysis.xlsx'

# Save each sheet separately
if not prod_data.empty:
    production_pivot.to_excel(excel_file, sheet_name='Production_by_Year', index=True)

with pd.ExcelWriter(excel_file, mode='a') as writer:
    if not prod_data.empty:
        stage_totals.to_frame('Total').to_excel(writer, sheet_name='Prod_by_Stage', index=True)
        production_stage_pivot.to_excel(writer, sheet_name='Prod_Stage_Region', index=True)
    if not active_data.empty:
        active_pivot.to_excel(writer, sheet_name='Active_by_Year', index=True)
        active_stage_totals.to_frame('Total').to_excel(writer, sheet_name='Active_by_Stage', index=True)
        active_stage_pivot.to_excel(writer, sheet_name='Active_Stage_Region', index=True)

print(f"💾 Saved: {excel_file}")

In [ ]:
# Create separate figures for each region
regions = set()
for results in analysis_results.values():
    if not results['production_summary'].empty:
        regions.update(results['production_summary']['Region'].unique())

regions = sorted(list(regions))

# 1. PRODUCTION PLOTS - One figure per region
for region in regions:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Production by Stage Over Time - {region}', fontsize=16, fontweight='bold')

    stages = ['mine', 'proc', 'cath', 'cell', 'pack', 'rec']

    for i, stage in enumerate(stages):
        ax = axes[i//3, i%3]

        # Plot each dataset for this stage and region
        for filename, results in analysis_results.items():
            if not results['production_summary'].empty:
                stage_region_data = results['production_summary'][
                    (results['production_summary']['Stage'] == stage) &
                    (results['production_summary']['Region'] == region)
                ]
                if not stage_region_data.empty:
                    ax.plot(stage_region_data['Year'], stage_region_data['Value'],
                           marker='o', label=filename.replace('.csv', ''), linewidth=2)

        ax.set_title(f'prod_{stage}')
        ax.set_xlabel('Year')
        ax.set_ylabel('Production')
        ax.legend(title='Scenario')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 2. ACTIVE FACILITIES PLOTS - One figure per region
for region in regions:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Active Facilities by Stage Over Time - {region}', fontsize=16, fontweight='bold')

    stages = ['mine', 'proc', 'cath', 'cell', 'pack', 'rec']

    for i, stage in enumerate(stages):
        ax = axes[i//3, i%3]

        # Plot each dataset for this stage and region
        for filename, results in analysis_results.items():
            if not results['active_summary'].empty:
                stage_region_data = results['active_summary'][
                    (results['active_summary']['Stage'] == stage) &
                    (results['active_summary']['Region'] == region)
                ]
                if not stage_region_data.empty:
                    ax.plot(stage_region_data['Year'], stage_region_data['Value'],
                           marker='s', label=filename.replace('.csv', ''), linewidth=2)

        ax.set_title(f'active_{stage}')
        ax.set_xlabel('Year')
        ax.set_ylabel('Number of Active Facilities')
        ax.legend(title='Scenario')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# SUMMARY TABLE
print(f"\n{'='*60}")
print("📋 SUMMARY COMPARISON TABLE")
print(f"{'='*60}")

summary_table = pd.DataFrame()
for filename, results in analysis_results.items():
    summary_table[filename] = {
        'Total Production': results['production_by_stage'].sum() if not results['production_by_stage'].empty else 0,
        'Total Active Facilities': results['active_by_stage'].sum() if not results['active_by_stage'].empty else 0,
        'Number of Stages (Production)': len(results['production_by_stage']) if not results['production_by_stage'].empty else 0,
        'Number of Stages (Active)': len(results['active_by_stage']) if not results['active_by_stage'].empty else 0,
    }

summary_df = pd.DataFrame(summary_table).T
print(summary_df)

In [ ]:
# Save individual analysis results
for filename, results in analysis_results.items():
    clean_filename = filename.replace('.csv', '_analysis.csv')

    # Save production summary
    if not results['production_summary'].empty:
        results['production_summary'].to_csv(f"{clean_filename}_production.csv", index=False)

    # Save active facilities summary
    if not results['active_summary'].empty:
        results['active_summary'].to_csv(f"{clean_filename}_active.csv", index=False)

print(f"\n💾 Individual analysis files saved!")
print(f"🎯 Analysis complete!")